In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install openai nest_asyncio tqdm pandas

In [3]:
import os
import json
import pandas as pd

# --- Configuration & Paths ---
BASE_DIR = '/content/drive/MyDrive/BengaliOffensive'
CSV_PATH = os.path.join(BASE_DIR, 'input.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'Runs')
OUTPUT_JSON_PATH = os.path.join(OUTPUT_DIR, 'prompts.json')

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Bengali User Prompt with Strict JSON Compliance Anchor ---
USER_PROMPT_TEMPLATE = "এই শব্দটি কি একজন বাংলা ভাষাভাষী মানুষের কাছে আপত্তিকর? শব্দ: '{WORD}'"

COMPLIANCE_ANCHOR = (
    " শুধুমাত্র নিচের JSON ফরম্যাটে উত্তর দাও: "
    '{"rating": <1-5>, "answer": <true/false>}। '
    "কোনো ব্যাখ্যা, যুক্তি বা অতিরিক্ত লেখা দিও না। "
    "মার্কডাউন কোড ব্লক ব্যবহার করো না।"
)

# --- Process Data & Build JSON ---
print("Loading Bengali word dataset...")
df = pd.read_csv(CSV_PATH)

prompts_array = []

for index, row in df.iterrows():
    serial = row['serial']
    word = row['word']
    # 'rating' column is intentionally ignored — LLM generates its own rating

    full_prompt = USER_PROMPT_TEMPLATE.format(WORD=word) + COMPLIANCE_ANCHOR

    prompts_array.append({
        "serial": int(serial),
        "word": word,
        "prompt": full_prompt
    })

# --- Save to Drive ---
with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(prompts_array, f, indent=4, ensure_ascii=False)

print("-" * 60)
print(f"✓ Success! Generated {len(prompts_array)} prompt objects.")
print(f"✓ File saved securely to: {OUTPUT_JSON_PATH}")

Loading Bengali word dataset...
------------------------------------------------------------
✓ Success! Generated 5 prompt objects.
✓ File saved securely to: /content/drive/MyDrive/BengaliOffensive/Runs/prompts.json


In [4]:
import os
import json
import asyncio
import nest_asyncio
import random
import re
import shutil
from google.colab import userdata
from tqdm.asyncio import tqdm
from openai import AsyncOpenAI

# Allow asyncio loops to run inside Colab
nest_asyncio.apply()

# --- Configuration & Paths ---
BASE_DIR = '/content/drive/MyDrive/BengaliOffensive'
INPUT_DIR = os.path.join(BASE_DIR, 'Runs')
PROMPTS_PATH = os.path.join(INPUT_DIR, 'prompts.json')
FINAL_RESPONSES_DIR = os.path.join(INPUT_DIR, 'responses')
TEMP_RESPONSES_DIR = '/content/responses'

os.makedirs(FINAL_RESPONSES_DIR, exist_ok=True)
os.makedirs(TEMP_RESPONSES_DIR, exist_ok=True)

# --- Models & Token Configuration ---
# max_tokens: small for direct responders, larger for reasoners (DeepSeek/Qwen3 think blocks)
MODELS = [
    {"id": "qwen/qwen3.7-flash",               "max_tokens": 512},
    {"id": "google/gemini-2.5-flash-lite",      "max_tokens": 80},
    {"id": "openai/gpt-oss-120b",               "max_tokens": 80},
    {"id": "openai/gpt-4o-mini",                "max_tokens": 80},
    {"id": "deepseek/deepseek-v4-flash",        "max_tokens": 512},
]

# --- Bengali System Prompt ---
SYSTEM_PROMPT = (
    "তুমি একজন বাংলা ভাষা ও সংস্কৃতি বিশেষজ্ঞ। "
    "তোমাকে একটি বাংলা শব্দ দেওয়া হবে। "
    "তুমি নির্ধারণ করবে এই শব্দটি একজন বাংলা ভাষাভাষী মানুষের কাছে আপত্তিকর কিনা। "
    "আপত্তিকরতার মাত্রা ১ থেকে ৫ এর মধ্যে নির্ধারণ করো, যেখানে ১ = সবচেয়ে কম আপত্তিকর এবং ৫ = সবচেয়ে বেশি আপত্তিকর।"
)

# 1. Load API Key & Initialize Client
try:
    os.environ["OPENROUTER_API_KEY"] = userdata.get('API_Ferdous')
    client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ.get("OPENROUTER_API_KEY"),
    )
    print("✓ OpenRouter API key loaded securely.")
except Exception as e:
    print("❌ Error: Could not find 'openrouter_api_key' in Colab Secrets.")

# 2. Load Prompts Dataset
with open(PROMPTS_PATH, 'r', encoding='utf-8') as f:
    prompts_data = json.load(f)

# 3. Initialize Output State Per Model
master_output_state = {}
for model_cfg in MODELS:
    model_id = model_cfg["id"]
    master_output_state[model_id] = []
    for item in prompts_data:
        master_output_state[model_id].append({
            "word": item['word'],
            "serial": item['serial'],
            "response": None
        })

print(f"✓ Loaded {len(prompts_data)} words for classification.")

# Lock for safe async writing
file_write_lock = asyncio.Lock()
sync_tracker = {'count': 0}


def get_safe_model_name(model_id):
    return model_id.replace("/", "_")


def extract_response(text):
    """Parse the LLM's JSON response. Returns (status, rating, answer).
    status: "SUCCESS" | "REFUSAL" | "ERROR"
    """

    # Purge DeepSeek/Qwen3 <think> reasoning blocks
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL | re.IGNORECASE)
    t = text.strip()

    # Catch Safety Refusals (English + Bengali) — these are REFUSAL, not ERROR
    t_upper = t.upper()
    if any(refusal in t_upper for refusal in ["SORRY", "AS AN AI", "CANNOT", "CAN'T", "UNABLE"]):
        return "REFUSAL", None, None
    if any(bn_refusal in t for bn_refusal in ["দুঃখিত", "আমি একটি এআই", "পারি না", "সক্ষম নই", "অক্ষম"]):
        return "REFUSAL", None, None

    # Strip markdown code fences if present
    t = re.sub(r'^```(?:json)?\s*', '', t)
    t = re.sub(r'\s*```$', '', t)
    t = t.strip()

    # Try to parse as JSON
    try:
        parsed = json.loads(t)
        rating = parsed.get("rating")
        answer = parsed.get("answer")

        # Validate rating is 1-5
        if isinstance(rating, (int, float)) and 1 <= int(rating) <= 5:
            rating = int(rating)
        else:
            return "ERROR", None, None

        # Validate answer is boolean
        if isinstance(answer, bool):
            pass
        elif isinstance(answer, str):
            if answer.lower() == "true":
                answer = True
            elif answer.lower() == "false":
                answer = False
            else:
                return "ERROR", None, None
        else:
            return "ERROR", None, None

        return "SUCCESS", rating, answer

    except (json.JSONDecodeError, AttributeError, TypeError):
        return "ERROR", None, None


async def process_word(model_cfg, word_index, word, full_prompt, local_path, drive_path, semaphore, pbar, total_tasks):
    async with semaphore:
        model_id = model_cfg["id"]
        max_retries = 3
        req_tokens = model_cfg["max_tokens"]

        status = "ERROR"
        rating = None
        answer = None
        refusal_count = 0
        error_count = 0

        for attempt in range(max_retries):
            try:
                response = await client.chat.completions.create(
                    model=model_id,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": full_prompt}
                    ],
                    temperature=0.0,
                    max_tokens=req_tokens
                )

                raw_content = response.choices[0].message.content
                response_text = raw_content.strip() if raw_content else ""

                status, rating, answer = extract_response(response_text)

                if status == "SUCCESS":
                    break  # True Success! No more retries needed.

                elif status == "REFUSAL":
                    refusal_count += 1
                    tqdm.write(f"🚫 Refusal for '{word}' (attempt {attempt + 1}/{max_retries}). Retrying...")
                    await asyncio.sleep(2)

                else:  # ERROR — malformed output
                    error_count += 1
                    tqdm.write(f"⚠️ Bad output for '{word}': '{response_text[:80]}...'. Retrying...")
                    await asyncio.sleep(2)

            except Exception as e:
                error_msg = str(e).lower()
                if "429" in error_msg or "rate limit" in error_msg:
                    sleep_time = (1.5 ** attempt) + random.uniform(0.5, 1.5)
                    if attempt > 0:
                        tqdm.write(f"🚦 Rate limit. Backing off {sleep_time:.1f}s...")
                    await asyncio.sleep(sleep_time)
                else:
                    error_count += 1
                    tqdm.write(f"❌ Error processing '{word}': {e}")
                    await asyncio.sleep(1)

        # --- DETERMINE FINAL STATUS AFTER ALL RETRIES ---
        if status == "SUCCESS":
            final_response = {
                "model": model_id,
                "rating": rating,
                "answer": answer,
                "status": "SUCCESS"
            }
        elif refusal_count >= max_retries:
            # Model consistently refused across all retries — save as REFUSAL
            tqdm.write(f"🚫 Confirmed REFUSAL for '{word}' after {max_retries} attempts. Saving as REFUSAL.")
            final_response = {
                "model": model_id,
                "rating": None,
                "answer": None,
                "status": "REFUSAL"
            }
        else:
            # Malformed output or API errors persisted — save as ERROR
            tqdm.write(f"🚨 Hard ERROR for '{word}' after {max_retries} attempts. Saving as ERROR.")
            final_response = {
                "model": model_id,
                "rating": None,
                "answer": None,
                "status": "ERROR"
            }

        # --- SAFE DISK WRITE LOCK (always saves — REFUSAL, ERROR, or SUCCESS) ---
        async with file_write_lock:
            master_output_state[model_id][word_index]["response"] = final_response

            # Immediate Local Save
            with open(local_path, 'w', encoding='utf-8') as f:
                json.dump(master_output_state[model_id], f, indent=4, ensure_ascii=False)

            sync_tracker['count'] += 1

            # Sync to Drive every 30 requests
            if sync_tracker['count'] % 30 == 0 or sync_tracker['count'] == total_tasks:
                shutil.copy2(local_path, drive_path)

        pbar.update(1)


async def run_pipeline():
    print("\n--- STARTING ASYNC BENGALI OFFENSIVE CLASSIFICATION (OPENROUTER) ---")

    semaphore = asyncio.Semaphore(50)

    for model_cfg in MODELS:
        model_id = model_cfg["id"]
        print(f"\n🚀 Currently Processing Model: {model_id}")

        safe_name = get_safe_model_name(model_id)
        drive_path = os.path.join(FINAL_RESPONSES_DIR, f"{safe_name}.json")
        local_path = os.path.join(TEMP_RESPONSES_DIR, f"{safe_name}.json")

        # --- RESUME LOGIC: Load existing Drive file & purge structurally invalid responses ---
        # SUCCESS, REFUSAL, and ERROR are all valid final states — only purge malformed entries
        if os.path.exists(drive_path):
            with open(drive_path, 'r', encoding='utf-8') as f:
                loaded_state = json.load(f)

            purged_count = 0
            for obj in loaded_state:
                if obj.get("response") is not None:
                    r = obj["response"]
                    resp_status = r.get("status")

                    if resp_status == "SUCCESS":
                        # Validate SUCCESS entries have valid rating + answer
                        rating_val = r.get("rating")
                        answer_val = r.get("answer")
                        if not (isinstance(rating_val, int) and 1 <= rating_val <= 5 and isinstance(answer_val, bool)):
                            obj["response"] = None
                            purged_count += 1

                    elif resp_status in ("REFUSAL", "ERROR"):
                        pass  # These are valid final states — keep them

                    else:
                        # No status field or unknown status — legacy/corrupted entry
                        obj["response"] = None
                        purged_count += 1

            master_output_state[model_id] = loaded_state

            if purged_count > 0:
                print(f"🧹 PURGED {purged_count} corrupted responses.")
            else:
                print(f"✓ Drive file loaded successfully.")

        # --- Queue only words with missing responses ---
        tasks_to_run = []
        for idx, item in enumerate(master_output_state[model_id]):
            if item.get("response") is None:
                original_prompt = prompts_data[idx]["prompt"]
                tasks_to_run.append((idx, item["word"], original_prompt))

        total_tasks = len(tasks_to_run)

        if total_tasks == 0:
            print(f"✓ {model_id} already 100% completed with clean data. Skipping.")
            continue

        print(f"Total tasks queued for {model_id}: {total_tasks}")
        short_name = model_id.split('/')[-1]

        sync_tracker['count'] = 0

        # Async stream processing
        with tqdm(total=total_tasks, desc=f"Querying {short_name}") as pbar:
            coroutines = [
                process_word(model_cfg, idx, word, prompt, local_path, drive_path, semaphore, pbar, total_tasks)
                for (idx, word, prompt) in tasks_to_run
            ]

            await asyncio.gather(*coroutines)

        # Final Drive sync
        if os.path.exists(local_path):
            shutil.copy2(local_path, drive_path)

        print(f"✓ {model_id} complete and safely backed up to Drive.")

    print("\n🎉 Pipeline Complete! All models generated with 100% data integrity.")

# Execute the async run
await run_pipeline()

✓ OpenRouter API key loaded securely.
✓ Loaded 5 words for classification.

--- STARTING ASYNC BENGALI OFFENSIVE CLASSIFICATION (OPENROUTER) ---

🚀 Currently Processing Model: qwen/qwen3.7-flash
Total tasks queued for qwen/qwen3.7-flash: 5


Querying qwen3.7-flash: 100%|██████████| 5/5 [00:18<00:00,  3.74s/it]


✓ qwen/qwen3.7-flash complete and safely backed up to Drive.

🚀 Currently Processing Model: google/gemini-2.5-flash-lite
Total tasks queued for google/gemini-2.5-flash-lite: 5


Querying gemini-2.5-flash-lite: 100%|██████████| 5/5 [00:00<00:00,  5.08it/s]


✓ google/gemini-2.5-flash-lite complete and safely backed up to Drive.

🚀 Currently Processing Model: openai/gpt-oss-120b
Total tasks queued for openai/gpt-oss-120b: 5


...

✓ openai/gpt-oss-120b complete and safely backed up to Drive.

🚀 Currently Processing Model: openai/gpt-4o-mini
Total tasks queued for openai/gpt-4o-mini: 5


Querying gpt-4o-mini: 100%|██████████| 5/5 [00:01<00:00,  4.05it/s]


✓ openai/gpt-4o-mini complete and safely backed up to Drive.

🚀 Currently Processing Model: deepseek/deepseek-v4-flash
Total tasks queued for deepseek/deepseek-v4-flash: 5


Querying deepseek-v4-flash: 100%|██████████| 5/5 [00:03<00:00,  1.59it/s]

✓ deepseek/deepseek-v4-flash complete and safely backed up to Drive.

🎉 Pipeline Complete! All models generated with 100% data integrity.


In [5]:
import os
import json
import pandas as pd

# --- Configuration & Paths ---
RESPONSES_DIR = '/content/drive/MyDrive/BengaliOffensive/Runs/responses'

def run_sanity_check():
    print("--- 🩺 BENGALI OFFENSIVE DATASET SANITY CHECK ---")

    if not os.path.exists(RESPONSES_DIR):
        print(f"❌ Error: Directory not found -> {RESPONSES_DIR}")
        return

    json_files = [f for f in os.listdir(RESPONSES_DIR) if f.endswith('.json')]

    if not json_files:
        print("❌ Error: No JSON files found in the directory.")
        return

    print(f"Found {len(json_files)} model files. Commencing deep scan...\n")

    report_data = []

    for filename in sorted(json_files):
        filepath = os.path.join(RESPONSES_DIR, filename)
        model_name = filename.replace('.json', '')

        status = "✅ PASS"
        total_words = 0
        success_count = 0
        refusal_count = 0
        error_count = 0
        missing_count = 0
        invalid_count = 0

        # 1. JSON Structural Validation
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except json.JSONDecodeError:
            print(f"🚨 FATAL: {filename} is corrupted and cannot be parsed!")
            report_data.append({
                "Model": model_name,
                "Status": "❌ CORRUPTED",
                "Total Words": "N/A",
                "Success": "N/A",
                "Refusals": "N/A",
                "Errors": "N/A",
                "Missing": "N/A",
                "Invalid": "N/A"
            })
            continue

        total_words = len(data)

        # 2. Deep Content Scan
        for item in data:
            resp = item.get("response")

            if resp is None:
                missing_count += 1
                continue

            resp_status = resp.get("status")

            if resp_status == "SUCCESS":
                rating = resp.get("rating")
                answer = resp.get("answer")
                if isinstance(rating, int) and 1 <= rating <= 5 and isinstance(answer, bool):
                    success_count += 1
                else:
                    invalid_count += 1
            elif resp_status == "REFUSAL":
                refusal_count += 1
            elif resp_status == "ERROR":
                error_count += 1
            else:
                invalid_count += 1

        # 3. Status Evaluation
        if missing_count > 0 or invalid_count > 0:
            status = "⚠️ FAIL"
        elif error_count > 0 or refusal_count > 0:
            status = "✅ DONE (with flags)"

        report_data.append({
            "Model": model_name,
            "Status": status,
            "Total Words": total_words,
            "Success": success_count,
            "Refusals": refusal_count,
            "Errors": error_count,
            "Missing": missing_count,
            "Invalid": invalid_count
        })

    # --- Display Report ---
    df_report = pd.DataFrame(report_data)
    print(df_report.to_string(index=False))
    print("\n" + "=" * 60)

    if all(df_report['Status'] == "✅ PASS"):
        print("🎉 ALL CLEAR! Your dataset has 100% data integrity.")
    else:
        print("🚨 ACTION REQUIRED: Some models failed the sanity check.")
        print("Re-run the pipeline cell. The self-healing resume logic will patch missing slots.")

# Execute the check
run_sanity_check()

--- 🩺 BENGALI OFFENSIVE DATASET SANITY CHECK ---
Found 5 model files. Commencing deep scan...

                       Model              Status  Total Words  Success  Refusals  Errors  Missing  Invalid
  deepseek_deepseek-v4-flash              ✅ PASS            5        5         0       0        0        0
google_gemini-2.5-flash-lite              ✅ PASS            5        5         0       0        0        0
          openai_gpt-4o-mini              ✅ PASS            5        5         0       0        0        0
         openai_gpt-oss-120b ✅ DONE (with flags)            5        3         0       2        0        0
          qwen_qwen3.7-flash              ✅ PASS            5        5         0       0        0        0

🚨 ACTION REQUIRED: Some models failed the sanity check.
Re-run the pipeline cell. The self-healing resume logic will patch missing slots.
